# PlastPrior: source data and chemical identities

This notebook presents source inspection, traceable record assembly, chemical identity resolution and aggregation by full InChIKey. QSAR evidence and scoring are outside the current implementation.

## Setup

Use the **PlastPrior (Python 3.12)** kernel, restart it, and run all cells from the project directory. The `README.md` describes installation, policies and validation. Inputs are read without modification; generated reports are written to `results/`.


In [ ]:
from pathlib import Path
from IPython.display import display

if not (Path.cwd() / "plastprior" / "inputs.py").is_file():
    raise RuntimeError("Run this notebook from the PlastPrior project directory.")

from plastprior.inputs import build_source_records, inspect_inputs
from plastprior.identity import resolve_identities

DATA_ROOT = Path("data")
REPORT_DIR = Path("results")

## Stage 1 — Input validation and ingestion

Inspect the declared source files and selected worksheets. Versions and snapshot dates are recorded metadata; **Not recorded** indicates a provenance gap. Parsed rows are source records, not unique chemicals, and must not be summed across overlapping inventories. Table roles distinguish inventory records, identity references, and literature links.

Stage 1 requires the source-file and table checks to pass. Auxiliary inputs are listed separately and receive **presence checks only**; their contents must be validated by the stages that use them.

In [ ]:
inspection = inspect_inputs(DATA_ROOT)
inspection.write_reports(REPORT_DIR)

display(inspection.sources[[
    "source", "filename", "version", "snapshot_date", "status"
]])
display(inspection.tables[[
    "source_id", "table_id", "table_role", "parsed_rows", "nonempty_rows", "columns", "status"
]])

**Review the inspection outcome.** Source errors stop this stage after the reports have been saved. Missing auxiliary inputs remain visible but do not invalidate source inspection. A successful inspection does not establish scientific validity or readiness for scoring.

Full provenance notes, references, observed SHA-256 fingerprints, sheet/header declarations, and diagnostics are retained in `results/source_inventory.csv`, `results/table_dimensions.csv`, and `results/supporting_inputs.csv`. These files are regenerated on each run. An observed fingerprint identifies file bytes; it does not independently certify a source version.

In [ ]:
display(inspection.supporting[["input_group", "status"]])
if not inspection.supporting_present:
    print("Supporting inputs are incomplete; see supporting_inputs.csv.")

for label, frame in [("Source files", inspection.sources), ("Source tables", inspection.tables)]:
    failures = frame.loc[frame["status"].ne("OK")]
    if not failures.empty:
        print(label)
        display(failures)

reader_notes = inspection.tables.loc[
    inspection.tables["reader_warnings"].ne(""), ["table_id", "reader_warnings"]
]
if not reader_notes.empty:
    print("Workbook-reader notes")
    display(reader_notes)

inspection.require_sources()
print("Stage 1 source inspection passed. Auxiliary content validation remains pending.")

## Stage 2 — Chemical identity resolution

**Source record assembly.**

Assemble the 11 inventory tables using the explicit mappings in `plastprior/sources.json`. PubChemLite, NORMAN SusDat and the LitChemPlast literature-link table are outside the candidate inventory. PlastChem uses isomeric SMILES when supplied, with canonical SMILES as the declared fallback. The historical MPLEACH rule requires both reported InChIKey and SMILES; excluded records remain in the exported table with reasons.

Availability counts refer to **included source records**, not unique chemicals or validated identifiers. Missingness is determined using the documented, case-sensitive token list after whitespace trimming. Original mapped values, including placeholders and unselected fallback values, are retained in `reported_fields_json`. An empty mapping means the field is not supplied by that source.

The full records, field mappings, summary and provenance manifest are saved under `results/stage2/`. Frozen-file fingerprints and record positions define record identifiers; native identifiers are retained where explicitly declared. This assembly step precedes chemical matching; its original records are retained throughout identity resolution.

In [ ]:
source_records = build_source_records(DATA_ROOT)
source_records.write_reports(REPORT_DIR / "stage2")

display(source_records.summary[[
    "source_id", "table_id", "parsed_records", "included_records", "excluded_records"
]])
display(source_records.summary[[
    "table_id", "included_records", "name_available", "cas_available",
    "smiles_available", "inchikey_available"
]])
excluded = source_records.records.loc[~source_records.records["included"]]
if excluded.empty:
    print("No inventory records were excluded under the declared inclusion rules.")
else:
    display(excluded.groupby(["table_id", "exclusion_reason"]).size().rename("records").reset_index())
print("Source record assembly complete; proceed to identity resolution below.")

**Resolve structures and retain all source links.** The versioned migration policy first uses acceptable reported SMILES, then full InChIKey lookup in PubChemLite, PlastChem and NORMAN, then CAS lookup only when eligible reference candidates agree on one full key. Reference mappings are explicit in `sources.json`. Tier A–D denotes route precedence, not a probability of correct identification.

Aggregation uses the full structure-derived InChIKey and preserves every member record. A parsed structure without a valid identity key is retained at record level but cannot enter the compound inventory. The non-isomeric QSAR key is an annotation only; it is not used to collapse stereoisomers. Name matching, desalting, tautomer normalisation, QSAR eligibility screening and scoring are not performed.

The summary separates matching routes from valid-key counts. Full record-level decisions, selected reference row identifiers, compound memberships, duplicate reference keys and CAS conflicts are exported under `results/stage2/`. See the README for the separate, fingerprinted historical regression command. A successful notebook execution alone does not establish agreement with the historical baseline.


In [ ]:
identities = resolve_identities(source_records, DATA_ROOT)
identities.write_reports(REPORT_DIR / "stage2")

for measure, count in identities.manifest["counts"].items():
    print(f"{measure.replace('_', ' ').capitalize()}: {count:,}")
display(identities.summary)
display(identities.reference_summary)
print("Stage 2 complete. Review unresolved records and audit reports before proceeding to QSAR integration.")
